# 00 — Business Understanding

> **Insiders Loyalty Program** · Customer Segmentation Project

---

Following the end-to-end ML project framework from *Hands-On Machine Learning with Scikit-Learn and PyTorch* (Géron, 2025), the first step is to **frame the problem clearly** before touching any data.

This notebook documents:
1. The business problem and objective
2. How the ML system will be used
3. What type of ML task this is
4. How we will measure success
5. The analytical plan (10-step roadmap)

## 1. Business Problem

An online retail company based in the United Kingdom sells gifts and homeware to customers worldwide. After years of operation, the company has accumulated a rich transaction history but has **no systematic way** to identify which customers are truly valuable.

The company wants to launch a loyalty program called **"Insiders"** — an exclusive club for high-value customers who will receive:
- Early access to new products
- Priority customer support
- Personalised discounts and promotions

**Core business questions:**
- Who are our best customers?
- How different are they from average customers?
- How much revenue is concentrated in the top segment?
- How many customers should be invited to the Insiders program?

## 2. How the Solution Will Be Used

The output of this project is an **actionable customer list** with segment labels. The marketing team will:
1. Use the Insiders cluster to send targeted campaigns
2. Use the at-risk cluster to trigger reactivation flows
3. Connect the SQLite output to a BI dashboard (Metabase, Power BI, etc.)
4. Re-run the pipeline monthly as new transactions arrive

The ML pipeline will also expose a **REST API** so that the CRM system can predict the segment for a new customer in real time.

## 3. Type of Machine Learning Task

Following Géron's classification framework (Chapter 1):

| Criterion | Answer |
|---|---|
| Supervised vs Unsupervised | **Unsupervised** — there are no predefined labels |
| Task type | **Clustering** — discover natural groups in the data |
| Learning mode | **Batch** — retrain on a snapshot of transactions |
| Algorithm family | **Centroid-based** (K-Means) + metric evaluation |

There is no "correct" answer about who is a high-value customer — the algorithm discovers the structure naturally from purchasing behaviour.

> **Key insight from the book:** *"Clustering algorithms try to detect groups of similar instances."* (Géron, Ch. 8). Our similarity criterion is the RFM profile: Recency, Frequency, and Monetary value.

## 4. Business Assumptions

Before building any model, we document assumptions agreed upon with the business team:

1. **Customer value** is best captured by combining Recency, Frequency, and Monetary behaviour (RFM framework).
2. **Cancelled orders** (invoices starting with `C`) should be excluded — they do not represent real purchasing intent.
3. **Customers without an ID** are anonymous and cannot be tracked, so they are excluded.
4. **Zero or negative quantities/prices** are data quality issues and should be removed.
5. The **Insiders cluster** is defined as the group with the best combination of: low recency, high frequency, and high monetary value.

## 5. Success Metrics

Since this is an unsupervised problem, we have no ground truth. We evaluate the clustering solution using three complementary metrics:

| Metric | Interpretation | Target |
|---|---|---|
| **Silhouette Score** | Measures how similar each point is to its own cluster vs others. Range: [-1, 1] | > 0.35 |
| **Davies-Bouldin Index** | Average ratio of within-cluster to between-cluster scatter. Lower = better | < 1.0 |
| **Calinski-Harabasz Score** | Ratio of between-cluster to within-cluster variance. Higher = better | > 100 |

Beyond technical metrics, the **business test** is the most important: do the clusters tell a coherent story that the marketing team can act on?

## 6. Analytical Roadmap

We follow a 10-step cycle adapted from Géron's end-to-end project checklist (Chapter 2):

| Step | Notebook | Description |
|---|---|---|
| 1. Data Description | `01_data_understanding` | Validate schema, dimensions, missing values, and data types |
| 2. Exploratory Analysis | `02_exploratory_analysis` | Distributions, outliers, Pareto analysis, hypotheses |
| 3. Feature Engineering | `03_feature_engineering` | Build RFM features, apply log transform, scale |
| 4. Modelling | `04_modeling_and_business_results` | Elbow curve, silhouette, multiple k values, best model |
| 5. Business Translation | `04_modeling_and_business_results` | Cluster profiles, Insiders definition, revenue analysis |
| 6. Deployment | `05_deployment_and_consumption` | API, SQLite export, BI dashboard connection |

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from insiders_loyalty_program.config import load_config

config = load_config(PROJECT_ROOT / 'configs' / 'project.toml')
print('Project:', config['project']['name'])
print('Problem type:', config['project']['problem_type'])
print('Family:', config['project']['family'])
print('Primary metric:', config['project']['primary_metric'])

## 7. Data Source

The dataset is publicly available on Kaggle:

- **URL:** https://www.kaggle.com/datasets/carrie1/ecommerce-data  
- **File:** `data/raw/Ecommerce.csv`
- **Size:** ~45 MB, 541,909 rows, 8 columns
- **Period:** December 2010 – December 2011 (1 year of transactions)
- **Granularity:** One row per transaction line item (not per order, not per customer)

The modelling pipeline will aggregate these rows into **one row per customer** using the RFM framework.

---
**Next notebook:** `01_data_understanding.ipynb` — load the raw data and inspect its structure.